**Neste arquivo ja realizamos diversas etapas antes de escolher o modelo do ExtraTreesClassifier e neste colab estou apenas otimizando o modelo escolhido, para ver as etapas anteriores vá neste outro colab:**

- https://colab.research.google.com/drive/1eqi1eYrkvQZoWOaStvpmMBms2S5Cvwxb

In [ ]:
# Imporytando bibliotecas
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_val_score
import matplotlib.pyplot as plt

In [ ]:
# Obtendo a base de dados
url = 'https://raw.githubusercontent.com/alvaroriz/datascience_datasets/refs/heads/main/spotfy_churn.csv'
base = pd.read_csv(url)
base.head()

In [ ]:
# Dividindo em base de dados em bases de trabalho e de validaçao
validacao = base.sample(frac=0.2, random_state=22)
base = base.drop(validacao.index)
print(f'Base de trabalho: {base.shape}')
print(f'Base de validação: {validacao.shape}')

In [ ]:
# Divisão da base de trabalho em treino e teste
base_train, base_test = train_test_split(base, test_size=0.2, random_state=22)
X_train = base_train.drop('is_churned', axis=1)
y_train = base_train['is_churned']

In [ ]:
base_train.shape, base_test.shape

# TREINO

In [ ]:
# Ordinal enconding em subscription_type
subscription_mapping = {'Free': 0, 'Student': 1, 'Family': 2,'Premium':3}
X_train['subscription_type'] = X_train['subscription_type'].map(subscription_mapping)
X_train['subscription_type'].value_counts()

In [ ]:
# One hot enconding com k -1 para gender e device_type
X_train = pd.get_dummies(X_train, columns=['gender', 'device_type'],drop_first=True)

In [ ]:
# Label enconding em Country
label_encoder = LabelEncoder()
X_train['country'] = label_encoder.fit_transform(X_train['country'])
X_train['country'].value_counts()

In [ ]:
X_train.shape

In [ ]:
# excluir offline_listening, subscription_type e user_id
X_train = X_train.drop(['offline_listening', 'subscription_type','user_id'], axis=1)
X_train.head()

In [ ]:
# Fazendo o Scaling para variáveis numéricas contínuas
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

# Pré-processador para treino
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['age', 'listening_time', 'songs_played_per_day', 'skip_rate', 'ads_listened_per_week']),
    ],
    remainder='passthrough'
)

# Aplicar FIT_TRANSFORM apenas no treino
X_train_processed = pd.DataFrame(
    preprocessor.fit_transform(X_train),
    columns=preprocessor.get_feature_names_out(),
    index=X_train.index
)
X_train = X_train_processed
X_train.head()

In [ ]:
# aplicando o smote na variável alvo que esta desbalanceada
#from imblearn.over_sampling import SMOTE
#smote = SMOTE(random_state=42)
#X_treino_balanced, y_treino_balanced = smote.fit_resample(X_train, y_train)
#base_treino_final = pd.concat([
#    pd.DataFrame(X_treino_balanced),
#    pd.DataFrame(y_treino_balanced)
#], axis=1)
#base_treino_final['is_churned'].value_counts()
#____________________________________________________________
# Tentando undersampling
from imblearn.under_sampling import RandomUnderSampler
rus = RandomUnderSampler(random_state=42)
X_treino_balanced, y_treino_balanced = rus.fit_resample(X_train, y_train)
base_treino_final = pd.concat([
    pd.DataFrame(X_treino_balanced),
    pd.DataFrame(y_treino_balanced)
], axis=1)
base_treino_final['is_churned'].value_counts()

# FAZENDO AS TRANSFORMAÇÕES NO TESTE

In [ ]:
# Pré-processamento do teste(mesma transformações aplicadas no treino)
X_test = base_test.drop('is_churned', axis=1)
y_test = base_test['is_churned']
X_test = X_test.drop(['offline_listening', 'subscription_type', 'user_id'], axis=1)
X_test = pd.get_dummies(X_test, columns=['gender', 'device_type'],drop_first=True)
X_test['country'] = label_encoder.transform(X_test['country'])  # Use transform, não fit_transform
# Aplicar apenas TRANSFORM no teste (usando o preprocessor já treinado)
X_test_processed = pd.DataFrame(
    preprocessor.transform(X_test),  # Apenas transform, não fit_transform
    columns=preprocessor.get_feature_names_out(),
    index=X_test.index
)
X_test = X_test_processed
X_test.head()

# MODELO

In [ ]:
# como o melhor modelo indicado pelo Pycaret foram 2 modelos de árvore(ExtraTreeClassifier e RandomForest com 80 e 78 de acurácia respectivamente)
# iremos tentar otimizar o ExtraTreesClassifier que é um modelo de árvore porém no Pycaret o resultado estava claramente com overfitting pois no treino a performance foi 80%
# e no teste e validação beirou o 70%, em modelos de árvore controlamos o overfitting/underfiting pela profundidade da árvore(max_depth)
# para fim de plotar o gráfico de curva de aprebdizagem no treino e no teste iremos treinar varios modelos com profundidades de arvores diferentes
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import GridSearchCV
profundidade_arvore = [5,10,11,13,14,15,20,25,30]
# dicionario
resultados_cross_validation_error = {}
# Aplicar o Extra Trees Classifier
for i in enumerate(profundidade_arvore):
  modelo_i = ExtraTreesClassifier(n_estimators=500,criterion='entropy',max_features=3,min_samples_split=5,max_depth=i[1], random_state=42)
  modelo_i.fit(X_treino_balanced, y_treino_balanced)
  resultado = cross_val_score(modelo_i, X_treino_balanced, y_treino_balanced, cv=10, scoring='accuracy').mean()
  resultados_cross_validation_error[i[1]] = 1 - resultado
  print(f"Acurácia do modelo {i[0]} no treino com árvore de  profundidade {i[1]}: {resultado:.4f}")

In [ ]:
# Param grid otimizado
#param_grid = {
#    'criterion': ['gini', 'entropy'],  # 'log_loss' é equivalente a 'entropy'
#    'n_estimators': [100, 250, 500],
#    'min_samples_split': [2, 5, 10, 15],  # Incluí valores menores
#    'min_samples_leaf': [1, 2, 4],  # Adicionei este parâmetro importante
#    'max_features': ['sqrt', 'log2', 0.5, 0.7],  # Formas mais inteligentes
#    'bootstrap': [True, False]  # Adicionei para variedade
#}

# Grid Search com mais validações
#grid_search = GridSearchCV(
#    modelo,
#    param_grid,
#    cv=5,  # Aumentei para 5 folds
#    n_jobs=-1,
#    scoring='accuracy',
#    verbose=1  # Para ver o progresso
#)

# Execute o grid search
#grid_search.fit(X_treino_balanced, y_treino_balanced)

# Resultados
#print("Melhores parâmetros:", grid_search.best_params_)
#print("Melhor score:", grid_search.best_score_)

In [ ]:
# Testando o modelo com várias pronfundidades de arvores diferentes
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import classification_report, accuracy_score
# Aplicar o Extra Trees Classifier
profundidade_arvore = [5,10,11,13,14,15,20,25,30]
resultados_teste_error = {}
# Aplicar o Extra Trees Classifier
for i in enumerate(profundidade_arvore):
  modelo_i = ExtraTreesClassifier(n_estimators=500,criterion='entropy',max_features=3,min_samples_split=5,max_depth=i[1], random_state=42)
  modelo_i.fit(X_treino_balanced, y_treino_balanced)
  cross_val_score(modelo_i, X_treino_balanced, y_treino_balanced, cv=5, scoring='accuracy').mean()
  y_pred = modelo_i.predict(X_test)
  # Calcular e mostrar a acurácia no teste
  acuracia_teste_i = accuracy_score(y_test, y_pred)
  resultados_teste_error[i[1]] = 1 - acuracia_teste_i
  print(f"Acurácia do modelo {i[0]} no teste com árvores de profundidade {i[1]}: {acuracia_teste_i:.4f}")

In [ ]:
# Gráfico de curva de aprendizagem(Erro X Pronfundidade da árvore), podemos perceber que entre o 10 e 15 temos o melhor modelo e definindo um limite de erro não maior que 5%
# podemos chegar a conclusão de que a melhor pronfundidade de árvore é 13
plt.figure(figsize=(12, 8))
# Plot com estilos melhorados
plt.plot(list(resultados_cross_validation_error.keys()),
         list(resultados_cross_validation_error.values()),
         marker='o', linestyle='-', color='blue', linewidth=2, markersize=8,
         label='Erro na Validação Cruzada', alpha=0.8)
plt.plot(list(resultados_teste_error.keys()),
         list(resultados_teste_error.values()),
         marker='s', linestyle='--', color='red', linewidth=2, markersize=8,
         label='Erro no Teste', alpha=0.8)
# Melhorias visuais
plt.xlabel('Profundidade da Árvore', fontsize=12, fontweight='bold')
plt.ylabel('Erro', fontsize=12, fontweight='bold')
plt.title('Erro vs Profundidade da Árvore', fontsize=14, fontweight='bold')
# Grade mais sutil
plt.grid(True, alpha=0.3, linestyle='-')
# Ajuste dos eixos para melhor visualização
plt.xlim(min(resultados_cross_validation_error.keys()) - 0.5,
         max(resultados_cross_validation_error.keys()) + 0.5)
# LIMITE DO EIXO Y DE 0 ATÉ 1
plt.ylim(0, 1)
# Adicionar linha no ponto ótimo
# Encontrar ponto ótimo baseado na diferença de erro > 0.05
profundidades = list(resultados_cross_validation_error.keys())
erro_validacao = list(resultados_cross_validation_error.values())
erro_teste_list = list(resultados_teste_error.values())

profundidade_otima = None
for prof, erro_val, erro_test in zip(profundidades, erro_validacao, erro_teste_list):
    if abs(erro_val - erro_test) > 0.05:
        profundidade_otima = prof
        break

# Fallback: usar mínimo da validação se não encontrar diferença
if profundidade_otima is None:
    erro_min_val = min(erro_validacao)
    profundidade_otima = profundidades[erro_validacao.index(erro_min_val)]

plt.axvline(x=profundidade_otima, color='green', linestyle=':', alpha=0.7,
           label=f'Profundidade Ótima: {profundidade_otima}')
# Legenda (agora inclui a linha vertical)
plt.legend(fontsize=11, framealpha=0.9, loc='best')
# Melhorar ticks
plt.xticks(list(resultados_cross_validation_error.keys()))
plt.tight_layout()
plt.show()

*Podemos perceber pela curva de aprendizagem que a profundidade ideal máxima das árvores do modelo é 13*

In [ ]:
# modelo sem overfitting com 13 de profundidade
melhor_modelo = ExtraTreesClassifier(n_estimators=500,criterion='entropy',max_features=3,min_samples_split=5,max_depth=13, random_state=22)
melhor_modelo.fit(X_treino_balanced, y_treino_balanced)
y_pred = melhor_modelo.predict(X_test)
# Calcular e mostrar a acurácia no teste
acuracia = accuracy_score(y_test, y_pred)
print(f"Acurácia do melhor modelo no teste com árvores de profundidade máxima 13: {acuracia:.4f}")

In [ ]:
!pip install shap

# VALIDAÇÃO

In [ ]:
# Pré-processamento da validação
X_validacao = validacao.drop('is_churned', axis=1)
y_validacao = validacao['is_churned']
X_validacao = X_validacao.drop(['offline_listening', 'subscription_type', 'user_id'], axis=1)
X_validacao = pd.get_dummies(X_validacao, columns=['gender', 'device_type'],drop_first=True)
X_validacao['country'] = label_encoder.transform(X_validacao['country'])  # Use transform, não fit_transform
# Aplicar apenas TRANSFORM no teste (usando o preprocessor já treinado)
X_validacao_processed = pd.DataFrame(
    preprocessor.transform(X_validacao),  # Apenas transform, não fit_transform
    columns=preprocessor.get_feature_names_out(),
    index=X_validacao.index
)
X_validacao = X_validacao_processed
X_validacao.head()

In [ ]:
validacao_pred = melhor_modelo.predict(X_validacao)
# Calcular e mostrar a acurácia na validação
acuracia_validacao = accuracy_score(y_validacao,validacao_pred)
print("\n" + "="*50 + "\n")
print(f"Acurácia do modelo: {acuracia_validacao:.4f}")
print("\n" + "="*50)
print(classification_report(y_validacao,validacao_pred))

In [ ]:
features_names = X_test.columns
features_names

# PERSISTÊNCIA DO MODELO